In [22]:
import numpy as np

In [23]:
class SoftmaxLogistic:
    
    def __init__(self , learning_rate = 0.01 , epochs = 1000 , batch_size = 32):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        self.weights = None
        self.intercept = None
        
    def softmax(self , z):
        z = z - np.max( z , axis = 1 , keepdims= True)
        exp_z = np.exp(z)
        
        return exp_z  / np.sum(exp_z , axis=1 , keepdims=True)
    
    def fit(self , X, y):
        
        samples , features = X.shape
        classes = len(np.unique(y))
        
        self.weights = np.zeros((features , classes))
        self.intercept = np.zeros(classes)
        
        for epoch in range(self.epochs):
            idx = np.random.permutation(samples)
            x_idx = X[idx]
            y_idx = y[idx]
            
            for start in range(0 , samples , self.batch_size):
                end = start + self.batch_size
                X_batch = x_idx[start:end]
                y_batch = y_idx[start:end]
                
                z = X_batch@self.weights + self.intercept
                probabilities = self.softmax(z)
                
                y_labeled = np.zeros_like(probabilities)
                y_labeled[np.arange(len(y_batch)),y_batch] = 1
                
                error = probabilities - y_labeled
                slope_weights = (X_batch.T @ error)/X_batch.shape[0]
                slope_intercept = np.sum(error , axis = 0)/X_batch.shape[0]
                
                self.weights -= self.learning_rate*slope_weights
                self.intercept -= self.learning_rate*slope_intercept
                
        return self
    
    
    def predict_probability(self , X):
        z = X@self.weights + self.intercept
        return self.softmax(z)

    def predict(self ,X):
        probabilities = self.predict_probability(X)
        return np.argmax(probabilities , axis = 1)

In [24]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(
    n_samples=1000,
    n_features=10,
    n_informative=8,
    n_redundant=0,
    n_classes=3,
    n_clusters_per_class=1,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [25]:
my_model = SoftmaxLogistic(
    learning_rate=0.01,
    epochs=1000,
    batch_size=32
)

my_model.fit(X_train, y_train)

y_pred = my_model.predict(X_test)

print(y_pred[:20])

[0 2 2 1 2 1 2 0 0 0 0 2 0 2 0 2 0 1 2 1]


In [26]:
from sklearn.linear_model import LogisticRegression

sk_model = LogisticRegression(
    max_iter=1000
)

sk_model.fit(X_train, y_train)

y_pred_sk = sk_model.predict(X_test)

In [27]:
from sklearn.metrics import accuracy_score, classification_report

print("From Scratch:")
print(accuracy_score(y_test, y_pred))

print("Sklearn:")
print(accuracy_score(y_test, y_pred_sk))

From Scratch:
0.905
Sklearn:
0.9


In [28]:
print("===== FROM SCRATCH =====")
print(classification_report(y_test, y_pred))

print("===== SKLEARN =====")
print(classification_report(y_test, y_pred_sk))

===== FROM SCRATCH =====
              precision    recall  f1-score   support

           0       0.94      1.00      0.97        66
           1       0.88      0.85      0.86        67
           2       0.89      0.87      0.88        67

    accuracy                           0.91       200
   macro avg       0.90      0.91      0.90       200
weighted avg       0.90      0.91      0.90       200

===== SKLEARN =====
              precision    recall  f1-score   support

           0       0.93      1.00      0.96        66
           1       0.88      0.85      0.86        67
           2       0.89      0.85      0.87        67

    accuracy                           0.90       200
   macro avg       0.90      0.90      0.90       200
weighted avg       0.90      0.90      0.90       200

